## Let's run intron clustering to annotate alternative splicing events given observed junctions in our cells 

In [1]:
# Load LeafletSC 
import LeafletSC
import os
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

# Define path that contains junction files
juncs_path = "/gpfs/commons/projects/knowles_singlecell_splicing/TabulaSenis/Leaflet/"
print("The junctions are loaded from the following path: " + juncs_path) 

# print the files in the path 
print("The files in the path are: " + str(os.listdir(juncs_path)))

# define path for saving the output data 
output_path = "/gpfs/commons/projects/knowles_singlecell_splicing/TabulaSenis/Leaflet/"

# we provide a gtf file for the human genome as well to make better sense of the junctions that are detected in cells
# please replace with the path to the gtf file on your system
gtf_file="/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/genome_files/gencode.vM19/genes/genes.gtf" 

The junctions are loaded from the following path: /gpfs/commons/projects/knowles_singlecell_splicing/TabulaSenis/Leaflet/
The files in the path are: ['astrocyte_18mo_intron_clusters_50_500000_2_20240501_bulk.gz', 'astrocyte_18mo_juncs.bed', 'F7_D045344_B009254_S151_junctions.bed', 'SRR2557112_junctions.bed']


### Let's first define some parameters for the analysis

In [ ]:
from LeafletSC.clustering.find_intron_clusters import main as find_intron_clusters
from LeafletSC.clustering.find_intron_clusters import visualize_local_events

# define additional parameters 
sequencing_type = "bulk"

# single cell junction file
junc_file="/commons/projects/knowles_singlecell_splicing/TabulaSenis/Leaflet/F7_D045344_B009254_S151_junctions.bed"

# ensure output files are to be saved in output_path 
output_file = output_path + "astrocyte_18mo_intron_clusters"
junc_bed_file= output_path + "astrocyte_18mo_juncs.bed" # you can load this file into IGV to visualize the junction coordinates 
min_intron_length = 50
max_intron_length = 500000
threshold_inc = 0.01 
min_junc_reads = 2
min_num_cells_wjunc = 1
keep_singletons = False # ignore junctions that do not share splice sites with any other junction (likely const)
junc_suffix = "*_junctions.bed" # depends on how you ran regtools 

### Run intron clustering 

In [2]:
all_juncs_df = find_intron_clusters(junc_files=junc_file, gtf_file=gtf_file, output_file=output_file, 
                       sequencing_type=sequencing_type, junc_bed_file=junc_bed_file, 
                       threshold_inc=threshold_inc, min_intron = min_intron_length,
                       max_intron=max_intron_length, min_junc_reads=min_junc_reads,
                       singleton=keep_singletons,
                       junc_suffix=junc_suffix, min_num_cells_wjunc=min_num_cells_wjunc,filter_shared_ss=True, 
                       run_notebook = True)

NameError: name 'find_intron_clusters' is not defined

In [4]:
# load bed file 
juncs_bed = pd.read_csv(junc_bed_file, sep="\t", header=None)
# remove columns 3 and 4 
juncs_bed = juncs_bed.drop(columns=[3, 4])
juncs_bed.columns = ["Chromosome", "Start", "End", "Strand", "junction_id", "Start_b", "End_b", "gene_id", "gene_name", "transcript_id", "exon_id"]

# cmobine with all_juncs_df and prep df for visualization 
dat_vis = all_juncs_df[["chrom", "chromStart", "chromEnd", "strand", "intron_length", "counts_total", "junction_id", "Cluster"]]
dat_vis = dat_vis.drop_duplicates()
# merge dat_vis with juncs_bed using all common columns 
dat_vis = dat_vis.merge(juncs_bed, how="left", on=["junction_id"])

NameError: name 'junc_bed_file' is not defined

In [3]:
dat_vis

NameError: name 'dat_vis' is not defined

In [ ]:
len(dat_vis.gene_id.unique())

In [ ]:
# viusalize junction 11_68822665_68829914

In [ ]:
#from LeafletSC.clustering.find_intron_clusters import visualize_junctions
junc_id = all_juncs_df.junction_id.sample(1).values[0]
j = junc_id
visualize_local_events(dat_vis, j, p_usage_ratio=True)

In [ ]:
# extract all the unique gene names in all_juncs_df
gene_names = dat_vis.gene_name.unique()
print(len(gene_names))

In [ ]:
# make plot less wide and icnrease font of everything
plt.figure(figsize=(6, 4))
sns.set(font_scale=1.5)
sns.set_style("whitegrid")
all_juncs_df.score.hist(bins=50)
# add xlab total reads in junction
plt.xlabel("Total reads in junction")
plt.ylabel("Frequency")
plt.title("Histogram of total reads in junctions")

In [ ]:
# plot number of junctions per intron cluster 
plt.figure(figsize=(6, 4))
sns.set(font_scale=1.5)
sns.set_style("whitegrid")
all_juncs_df.Cluster.value_counts().hist(bins=50)
# add xlab total reads in junction
plt.xlabel("Number of junctions per intron cluster")
plt.ylabel("Frequency")
plt.title("Histogram of number of junctions per intron cluster")

In [ ]:
# find cluster with more than 10 junctions in it
clusters = all_juncs_df.Cluster.value_counts()
clusters = clusters[clusters > 6]
clusters

In [ ]:
all_juncs_df[all_juncs_df["Cluster"] == 81]

In [ ]:
j = "11_120562545_120562654"
visualize_local_events(dat_vis, j, p_usage_ratio=True)

In [ ]:
# get average number of junctions per cluster here 
avg_juncs_per_cluster = all_juncs_df.groupby("Cluster").size().mean()
print("The average number of junctions per cluster is: " + str(avg_juncs_per_cluster))